In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

# GoogleドライブとColab本体の空き容量を確認（80〜90GB級の超巨大モデル用）
_, _, drive_free = shutil.disk_usage('/content/drive/MyDrive')
_, _, local_free = shutil.disk_usage('/')
drive_free_gb = drive_free / (1024**3)
local_free_gb = local_free / (1024**3)
print(f"💾 Googleドライブ空き容量: {drive_free_gb:.1f} GB / Colab本体SSD空き容量: {local_free_gb:.1f} GB")

# 90GB級モデルを保存するため、100GB以上空いている方を優先選択
if drive_free_gb >= 100:
    os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'
    print("✅ モデルはGoogleドライブ (/content/drive/MyDrive/ollama_models) に永続保存されます。")
else:
    os.environ['OLLAMA_MODELS'] = '/root/.ollama/models'
    print("⚡ Colab本体の高速SSD (/root/.ollama/models) にモデルを保存します。")


In [ ]:
import os
import subprocess
import time
import shutil
import re

# 破損ファイルや過去のゴミをクリーンアップ
for path in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/content/cloudflared.deb"]:
    if os.path.exists(path):
        try:
            if os.path.isdir(path):
                shutil.rmtree(path)
            else:
                os.remove(path)
        except Exception:
            pass

def download_file(url, filepath):
    try:
        subprocess.run(["curl", "-L", "-o", filepath, url], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

# 0. 依存パッケージ (zstd + G4 Blackwell GPU自動検出用pciutils) をインストール
print("📦 依存パッケージ (zstd, pciutils) をインストール中...")
subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "zstd", "pciutils"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 1. Ollama のインストール (G4 / RTX PRO 6000 Blackwell アーキテクチャを自動認識)
print("📦 Ollama をインストール中...")
!curl -fsSL https://ollama.com/install.sh | sh

# 2. cloudflared のインストール
print("📦 Cloudflare Tunnel を準備中...")
cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
success_cf = download_file(cf_url, "/content/cloudflared.deb")

if success_cf and os.path.exists("/content/cloudflared.deb") and os.path.getsize("/content/cloudflared.deb") > 5000000:
    !dpkg -i /content/cloudflared.deb > /dev/null 2>&1
    print("✅ Cloudflare Tunnel の準備完了")
else:
    raise RuntimeError("❌ Cloudflare Tunnel のダウンロードに失敗しました。")

# 3. G4 GPU (96GB VRAM) を1モデルで限界まで使い切るパラメーター設定
print("🚀 Ollama サーバーを G4 GPU (90GB級超巨大モデル仕様) で起動中...")
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '65536'     # ★85GBモデル＋64Kコンテキストで約92GB VRAMを使用（96GB枠にピタリ収まる）
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'        # ★Flash Attention必須（90GB級モデルのVRAM溢れ防止＆高速化）
os.environ['OLLAMA_KV_CACHE_TYPE'] = 'q8_0'       # ★KVキャッシュを8bit化（残り約10GBのVRAMに64K文脈を収める）
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'      # ★重要：80〜90GB級モデルなので同時ロード数は必ず「1」に固定
os.environ['OLLAMA_NUM_PARALLEL'] = '1'           # ★1つの推論に96GB全てのVRAMと帯域を集中
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'            # ★一度ロードした巨大モデルをVRAMに常駐

subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

# 4. G4 GPU (96GB VRAM) 限界突破モデル（VRAM 75〜93GB 使用クラス）
# ※1つ65〜85GBあるため、ディスク容量の都合上、同時にpullするのは1〜2個がおすすめです
MODELS = [
    "qwen3-coder-next:q8_0", # ★約85GB (推論時約92GB): 80B MoEの8bit超高精度版！劣化ゼロ＆実効3Bで爆速の最強エージェント
    "gemma4:31b-it-bf16",    # ★約63GB (推論時約75GB): Google Gemma 4 31Bの非圧縮フル精度(BF16)版！量子化なしのGoogle最高画質・最高知能
    # "gpt-oss:120b",        # ★約65GB (推論時約80GB): OpenAI公式の120B超巨大オープンモデル（ツール＆思考推論対応）
    # "llama4:scout",        # ★約67GB (推論時約82GB): Meta最新 109B MoE マルチモーダルモデル
    # "mistral-large:123b",  # ★約73GB (推論時約88GB): Mistral最上位 123B 超大型モデル
]

for model in MODELS:
    print(f"\n📥 超大型モデル ({model}) を準備中（60〜85GBあるため数分かかります）...")
    !ollama pull {model}

# 5. Cloudflare Tunnel を起動してURLを発行
print("\n🌐 Cloudflare Tunnel を起動中...")
log_path = "/content/cloudflared.log"
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434",
     "--http-host-header", "localhost:11434"],
    stdout=open(log_path, "w"), stderr=subprocess.STDOUT
)

url = None
for _ in range(60):
    time.sleep(1)
    if os.path.exists(log_path):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            url = m.group(0)
            break

if url:
    print("\n" + "="*65)
    print("🎉 G4 GPU (90GB級モンスターモデル) 準備完了！Base URL に貼ってください：")
    print(f"👉 {url}")
    print("="*65)
    print("📌 使えるモデルID一覧（Zoo CodeのModel欄に入力）：")
    for m in MODELS:
        print(f"   • {m}")
    print("\n⚠️ このノートブックを閉じるとURLは無効になります。")
else:
    print("❌ URLを取得できませんでした。/content/cloudflared.log を確認してください。")
